In [24]:
import numpy as np
import json
import sys
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sem_proj.data.datasets import BoasDataset
from sem_proj.data.preprocessing import PreprocessingConfig
from sem_proj.models.model_factory import SSLEpochTransformerConv1D_v2, SSLClassifierHead, SSLLinearProbing
from sem_proj.data.boa_loader import build_pid_mappings

CHECKPOINT_LEOMED_DIR = PROJECT_ROOT / "checkpoints_leomed"
JSON_DIR = PROJECT_ROOT / "reports" / "metrics"
TARGET_DIR = PROJECT_ROOT / "plots"
SPLITS_FILE = PROJECT_ROOT / "data" / "processed" / "data_splits_70_15_15.json"
CONFIG_DIR = PROJECT_ROOT / "configs" / "preprocess"

In [16]:
def forwardpass_testset(model, dataloader, device):
    model.eval()
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            x, y = batch
            x = x.to(device)  # (B, C, T)
            y = y.to(device)  # (B,)

            output = model(x)  # (B, num_classes)

            preds = output.argmax(dim=-1)  # (B,)
            total_correct += (preds == y).sum().item()
            total_samples += y.numel()

            all_preds.append(preds.cpu().numpy().flatten())
            all_labels.append(y.cpu().numpy().flatten())
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    accuracy = total_correct / total_samples if total_samples > 0 else 0.0

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    return accuracy, macro_f1, per_class_f1

In [17]:
preprocess_config = PreprocessingConfig.from_yaml(CONFIG_DIR / "notch_bandpass_resample_znorm.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open(SPLITS_FILE, 'r') as f:
    splits = json.load(f)
test_nights = splits['test_subjects']

In [26]:
exp = "ctxfree_stage1_"
names_finetuned = [exp + f"p{p}_ssl_finetuning_stronger_MLP_v1" for p in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]]
names_fullysuperv = [exp + f"p{p}_fully_supervised_stronger_MLP_v1" for p in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]]
names_linprob = [exp + f"p{p}_ssl_finetuning_linearprobing" for p in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]]

print("Names:")
print("Finetuned:", names_finetuned)
print("Fully Supervised:", names_fullysuperv)
print("Linear Probe:", names_linprob)

test_metrics_file_finetuned = JSON_DIR / "ctxfree_finetuning_results_test_forward_pass_stronger_MLP_v1.json"
test_metrics_file_fullysuperv = JSON_DIR / "ctxfree_fullysupervised_results_test_forward_pass_stronger_MLP_v1.json"
test_metrics_file_linprob = JSON_DIR / "ctxfree_linearprobe_results_test_forward_pass.json"


Names:
Finetuned: ['ctxfree_stage1_p0.01_ssl_finetuning_stronger_MLP_v1', 'ctxfree_stage1_p0.05_ssl_finetuning_stronger_MLP_v1', 'ctxfree_stage1_p0.1_ssl_finetuning_stronger_MLP_v1', 'ctxfree_stage1_p0.2_ssl_finetuning_stronger_MLP_v1', 'ctxfree_stage1_p0.5_ssl_finetuning_stronger_MLP_v1', 'ctxfree_stage1_p1.0_ssl_finetuning_stronger_MLP_v1']
Fully Supervised: ['ctxfree_stage1_p0.01_fully_supervised_stronger_MLP_v1', 'ctxfree_stage1_p0.05_fully_supervised_stronger_MLP_v1', 'ctxfree_stage1_p0.1_fully_supervised_stronger_MLP_v1', 'ctxfree_stage1_p0.2_fully_supervised_stronger_MLP_v1', 'ctxfree_stage1_p0.5_fully_supervised_stronger_MLP_v1', 'ctxfree_stage1_p1.0_fully_supervised_stronger_MLP_v1']
Linear Probe: ['ctxfree_stage1_p0.01_ssl_finetuning_linearprobing', 'ctxfree_stage1_p0.05_ssl_finetuning_linearprobing', 'ctxfree_stage1_p0.1_ssl_finetuning_linearprobing', 'ctxfree_stage1_p0.2_ssl_finetuning_linearprobing', 'ctxfree_stage1_p0.5_ssl_finetuning_linearprobing', 'ctxfree_stage1_p1.0_

In [19]:
### start with finetuned models (share the same architecture as fully supervised) ###
encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
head = SSLClassifierHead(d_model=encoder.d_model, dropout=0.2, num_classes=5)
model = nn.Sequential(encoder, head)
model.to(device)

test_metrics_finetuned_dict = {}
if test_metrics_file_finetuned.exists():
    with open(test_metrics_file_finetuned, 'r') as f:
        test_metrics_finetuned_dict = json.load(f)
else:
    test_ds = BoasDataset(
        subjects=test_nights,
        mode='headband',
        transform_hb=None,
        preprocess_config=preprocess_config,
        use_cache=True
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=512,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        persistent_workers=False
    )
    for name in names_finetuned:
        checkpoint_path = CHECKPOINT_LEOMED_DIR / name / "best_model.pt"
        checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint_dict['model_state_dict'])
        model.to(device)

        test_acc, test_mf1, test_perclass_f1 = forwardpass_testset(model, test_dl, device)
        test_metrics_finetuned_dict[name] = {
            "test_acc": test_acc,
            "test_mf1": test_mf1,
            "test_perclass_f1": test_perclass_f1.tolist()
        }
    with open(test_metrics_file_finetuned, 'w') as f:
        json.dump(test_metrics_finetuned_dict, f, indent=4)


Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_63004\3460719607.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [21]:
### continue with fully supervised models (same architecture as finetuned) ###
encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
head = SSLClassifierHead(d_model=encoder.d_model, dropout=0.2, num_classes=5)
model = nn.Sequential(encoder, head)
model.to(device)

test_metrics_fullysuperv_dict = {}
if test_metrics_file_fullysuperv.exists():
    with open(test_metrics_file_fullysuperv, 'r') as f:
        test_metrics_fullysuperv_dict = json.load(f)
else:
    test_ds = BoasDataset(
        subjects=test_nights,
        mode='headband',
        transform_hb=None,
        preprocess_config=preprocess_config,
        use_cache=True
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=512,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        persistent_workers=False
    )
    for name in names_fullysuperv:
        checkpoint_path = CHECKPOINT_LEOMED_DIR / name / "best_model.pt"
        checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint_dict['model_state_dict'])
        model.to(device)

        test_acc, test_mf1, test_perclass_f1 = forwardpass_testset(model, test_dl, device)
        test_metrics_fullysuperv_dict[name] = {
            "test_acc": test_acc,
            "test_mf1": test_mf1,
            "test_perclass_f1": test_perclass_f1.tolist()
        }
    with open(test_metrics_file_fullysuperv, 'w') as f:
        json.dump(test_metrics_fullysuperv_dict, f, indent=4)

Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_63004\4116833142.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [27]:
### now finilize it with linear probed models (slightly different architecture) ###
encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
head = SSLLinearProbing(d_model=encoder.d_model, num_classes=5)
model = nn.Sequential(encoder, head)
model.to(device)

test_metrics_linprob_dict = {}
if test_metrics_file_linprob.exists():
    with open(test_metrics_file_linprob, 'r') as f:
        test_metrics_linprob_dict = json.load(f)
else:
    test_ds = BoasDataset(
        subjects=test_nights,
        mode='headband',
        transform_hb=None,
        preprocess_config=preprocess_config,
        use_cache=True
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=512,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        persistent_workers=False
    )
    for name in names_linprob:
        checkpoint_path = CHECKPOINT_LEOMED_DIR / name / "best_model.pt"
        checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint_dict['model_state_dict'])
        model.to(device)

        test_acc, test_mf1, test_perclass_f1 = forwardpass_testset(model, test_dl, device)
        test_metrics_linprob_dict[name] = {
            "test_acc": test_acc,
            "test_mf1": test_mf1,
            "test_perclass_f1": test_perclass_f1.tolist()
        }
    with open(test_metrics_file_linprob, 'w') as f:
        json.dump(test_metrics_linprob_dict, f, indent=4)

Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_63004\1333363943.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [ ]:
# plot test performance curves (varying data fraction)
mf1_finetuned = []
mf1_fullysuperv = []
mf1_linprob = []
for key in test_metrics_finetuned_dict.keys():
    nested_dict = test_metrics_finetuned_dict[key]
    mf1_score = nested_dict['test_mf1']
    mf1_finetuned.append(mf1_score)
for key in test_metrics_fullysuperv_dict.keys():
    nested_dict = test_metrics_fullysuperv_dict[key]
    mf1_score = nested_dict['test_mf1']
    mf1_fullysuperv.append(mf1_score)
for key in test_metrics_linprob_dict.keys():
    nested_dict = test_metrics_linprob_dict[key]
    mf1_score = nested_dict['test_mf1']
    mf1_linprob.append(mf1_score)

p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage
plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.plot(p, mf1_linprob, marker='s', linewidth=2, markersize=8, label='SSL Pretrain + Linear Probe', color='blue')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxfree_test_forward_pass_finetuning_vs_fullysupervised_vs_linearprobing_varying_p.pdf', dpi=300, bbox_inches='tight')

In [29]:
w_finetuned, n1_finetuned, n2_finetuned, n3_finetuned, rem_finetuned = [], [], [], [], []
w_fullysuperv, n1_fullysuperv, n2_fullysuperv, n3_fullysuperv, rem_fullysuperv = [], [], [], [], []
w_linprob, n1_linprob, n2_linprob, n3_linprob, rem_linprob = [], [], [], [], []
for key in test_metrics_finetuned_dict.keys():
    nested_dict = test_metrics_finetuned_dict[key]
    per_class_f1 = nested_dict['test_perclass_f1']
    w_finetuned.append(per_class_f1[0])
    n1_finetuned.append(per_class_f1[1])
    n2_finetuned.append(per_class_f1[2])
    n3_finetuned.append(per_class_f1[3])
    rem_finetuned.append(per_class_f1[4])
for key in test_metrics_fullysuperv_dict.keys():
    nested_dict = test_metrics_fullysuperv_dict[key]
    per_class_f1 = nested_dict['test_perclass_f1']
    w_fullysuperv.append(per_class_f1[0])
    n1_fullysuperv.append(per_class_f1[1])
    n2_fullysuperv.append(per_class_f1[2])
    n3_fullysuperv.append(per_class_f1[3])
    rem_fullysuperv.append(per_class_f1[4])
for key in test_metrics_linprob_dict.keys():
    nested_dict = test_metrics_linprob_dict[key]
    per_class_f1 = nested_dict['test_perclass_f1']
    w_linprob.append(per_class_f1[0])
    n1_linprob.append(per_class_f1[1])
    n2_linprob.append(per_class_f1[2])
    n3_linprob.append(per_class_f1[3])
    rem_linprob.append(per_class_f1[4])


# plot 2x3 grid
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()
stages = ['Wake', 'N1', 'N2', 'N3', 'REM']
finetuned_data = [w_finetuned, n1_finetuned, n2_finetuned, n3_finetuned, rem_finetuned]
fullysuperv_data = [w_fullysuperv, n1_fullysuperv, n2_fullysuperv, n3_fullysuperv, rem_fullysuperv]
linprobe_data = [w_linprob, n1_linprob, n2_linprob, n3_linprob, rem_linprob]
for idx, (ax, stage, fine, fully, lin) in enumerate(zip(axes_flat[:5], stages, finetuned_data, fullysuperv_data, linprobe_data)):
    ax.set_title(f'{stage}', fontsize=14, fontweight='bold')
    ax.plot(p, fine, marker='x', linewidth=2, markersize=7, color='green', label='SSL Pretrain + FT')
    ax.plot(p, fully, marker='o', linewidth=2, markersize=7, color='red', label='Supervised from Scratch')
    ax.plot(p, lin, marker='s', linewidth=2, markersize=7, color='blue', label='SSL Pretrain + Linear Probe')
    ax.set_xlabel('Percentage of labeled training data (%)', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.tick_params(axis='both', labelsize=11)
    ax.grid(True)
    if idx == 0:
        ax.legend(fontsize=10)
axes_flat[5].axis('off')  # leave last subplot empty
plt.tight_layout()
plt.savefig(TARGET_DIR / 'finetuned_vs_fullysupervised_vs_linprobed_perclassf1_ctxfree_test_forward_pass_grid.pdf', dpi=300, bbox_inches='tight')
